In [ ]:
"""
quasi-identifier set for the victim side:
gender (SEX) → gender_code
birth_year (derived from AGEP) → birth_year / age_at_year_end
race (RAC1P) → race_code
ethnicity (HISP) → ethnic_code
geography → scoped to Charlotte-area PUMA(s), reported as "CHARLOTTE"
"""

In [1]:
!pip install sdv

In [78]:
import requests
from google.colab import userdata
import pandas as pd
import pyarrow
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata
from sdv.evaluation import evaluate_quality
from sdmetrics.column_pairs import ContingencySimilarity
from sdv.metadata import SingleTableMetadata



In [15]:
url = "https://api.census.gov/data/2024/acs/acs5/pums"
params = {
    "get":"AGEP,SEX",
    "for":"public use microdata area:03101,03102,03103,03104,03105",
    "in": "state:37",
    "key": userdata.get('CENSUS')

}
response = requests.get(url, params)
print(response.status_code)

200
[["AGEP","SEX","state","public use microdata area"],
["90","2","37","03104"],
["69","1","37","03102"],
["82","1","37","03101"],
["19","2","37","03103"],
["17","1","37","03102"],
["17","1","37","03104"],
["25","1","37","03102"],
["37","2","37","03103"],
["18","2","37","03102"],
["20","2","37","03105"],
["14","2","37","03103"],
["22","2","37","03103"],
["19","1","37","03103"],
["23","2","37","03103"],
["17","1","37","03102"],
["61","1","37","03104"],
["18","1","37","03103"],
["19","2","37","03101"],
["19","1","37","03103"],
["61","1","37","03101"],
["16","1","37","03101"],
["21","2","37","03105"],
["40","1","37","03101"],
["16","1","37","03102"],
["21","2","37","03102"],
["65","1","37","03102"],
["18","1","37","03101"],
["17","1","37","03102"],
["65","1","37","03102"],
["18","1","37","03102"],
["66","2","37","03104"],
["18","1","37","03103"],
["19","1","37","03103"],
["22","2","37","03103"],
["19","2","37","03103"],
["21","2","37","03103"],
["18","2","37","03103"],
["35","2","37","03

In [20]:
data = response.json()
print(data[0])   # header row
print(data[1])
print(data[2])
len(data)

['AGEP', 'SEX', 'state', 'public use microdata area']
['90', '2', '37', '03104']
['69', '1', '37', '03102']


29897

In [21]:
df = pd.DataFrame(data[1:], columns=data[0])
print(df)

      AGEP SEX state public use microdata area
0       90   2    37                     03104
1       69   1    37                     03102
2       82   1    37                     03101
3       19   2    37                     03103
4       17   1    37                     03102
...    ...  ..   ...                       ...
29891   47   2    37                     03104
29892   49   1    37                     03104
29893   18   1    37                     03104
29894   15   2    37                     03104
29895   15   2    37                     03104

[29896 rows x 4 columns]


In [3]:
def fetch_pums_data(key, pumas, year=2024, state_code= 37, get_vars = "AGEP,SEX,RAC1P,HISP,PWGTP"):
  # pull pums from census using api; current default to NC from acs period 2020-24
  # return df of pums
  url = f"https://api.census.gov/data/{year}/acs/acs5/pums"
  pumas = ','.join(pumas)
  params = {
    "get":get_vars,
    "for":f"public use microdata area:{pumas}",
    "in": f"state:{state_code}",
    "key": key
  }
  response = requests.get(url, params)
  # check for screw ups
  response.raise_for_status()
  data = response.json()
  pums = pd.DataFrame(data[1:], columns=data[0])
  # chang e dtype of the codes to int
  pums[["AGEP", "PWGTP"]] = pums[["AGEP", "PWGTP"]].apply(pd.to_numeric)
  return pums

In [41]:
pumas = ["03101", "03102", "03103", "03104", "03105"]
key = userdata.get('CENSUS')

df = fetch_pums_data(key, pumas)
print(df.shape)
print(df['public use microdata area'].unique())

(29896, 7)
['03104' '03102' '03101' '03103' '03105']


In [42]:
print(df.dtypes)

AGEP                          int64
SEX                          object
RAC1P                        object
HISP                         object
PWGTP                         int64
state                        object
public use microdata area    object
dtype: object


In [43]:
df.to_parquet("pums_charlotte.parquet")

In [47]:
df_s = pd.read_parquet("/content/pums_charlotte.parquet")

In [48]:
df_s = df_s.drop(columns=['state'])

In [49]:
df = df_s.loc[df_s.index.repeat(df_s['PWGTP'])]


In [50]:
df.shape

(670892, 6)

In [51]:
df.drop(columns='PWGTP', inplace=True)
df_s.drop(columns='PWGTP', inplace=True)
print(df.shape)
print(df_s.shape)

(670892, 5)
(29896, 5)


## syn data

In [80]:
# making the syn data
metadata_old = SingleTableMetadata()
metadata_old.detect_from_dataframe(data=df_s)
gc = GaussianCopulaSynthesizer(metadata_old)
gc.fit(df)
syn_df = gc.sample(num_rows=15000)


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [81]:
metadata_old.to_dict()

{'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'columns': {'AGEP': {'sdtype': 'numerical'},
  'SEX': {'sdtype': 'categorical'},
  'RAC1P': {'sdtype': 'categorical'},
  'HISP': {'sdtype': 'categorical'},
  'public use microdata area': {'sdtype': 'categorical'}}}

In [82]:
print(syn_df.shape)
print(syn_df.dtypes)
print(syn_df.head())

(15000, 5)
AGEP                          int64
SEX                          object
RAC1P                        object
HISP                         object
public use microdata area    object
dtype: object
   AGEP SEX RAC1P HISP public use microdata area
0    48   1     1   01                     03101
1    36   1     2   11                     03101
2    22   1     1   01                     03103
3    32   2     2   01                     03103
4    34   2     6   01                     03104


In [65]:
# age
print(syn_df['AGEP'].describe())


count    15000.000000
mean        36.410467
std         21.962584
min          1.000000
25%         18.000000
50%         34.000000
75%         53.000000
max         94.000000
Name: AGEP, dtype: float64


In [66]:
# sex
print(syn_df['SEX'].min(), syn_df['SEX'].max())

1 2


In [67]:
#race-- 1-9
print(syn_df['RAC1P'].min(), syn_df['RAC1P'].max())

1 9


In [68]:
# hisp -- 01 to 24
print(syn_df['HISP'].min(), syn_df['HISP'].max())

01 24


In [90]:
from sdmetrics.column_pairs import ContingencySimilarity
import itertools

categorical_cols = ['SEX', 'RAC1P', 'HISP', 'public use microdata area']

results = []
for col1, col2 in itertools.combinations(categorical_cols, 2):
    score = ContingencySimilarity.compute(
        real_data=df_s[[col1, col2]],
        synthetic_data=syn_df[[col1, col2]]
    )
    results.append({'col1': col1, 'col2': col2, 'score': score})

import pandas as pd
pd.DataFrame(results)

,col1,col2,score
0,SEX,RAC1P,0.868510
1,SEX,HISP,0.950847
2,SEX,public use microdata area,0.929630
3,RAC1P,HISP,0.750891
4,RAC1P,public use microdata area,0.772431
5,HISP,public use microdata area,0.888456


In [92]:
syn_df.to_parquet("synthetic_charlotte_pop.parquet")